# 03. Build the pooled arm from LLM-coded controls

Combines the LLM-coded control severities with the AI arm into `pooled_coded.csv`, which `04`
section E reads. Handles two data issues found during validation:
- duplicate opinions (same text under different captions) are de-duplicated on the opinion text
- ineligible cases (criminal community-control, licensing, bar discipline) are dropped (`llm_eligible==False`)

Diagnostics print row counts before and after the merge and the concat, with an unmatched-key check.

Inputs (in `../data/coded/`): `controls_all_fulltext.csv`, `controls_llm_coded.csv`, `analysis_data_coded.csv`
Output: `../data/coded/pooled_coded.csv`

In [1]:
import pandas as pd, numpy as np

# ---------- functions (defined at top) ----------
def field_from_nos(nos):
    """Coarse legal-field bucket from a CourtListener Nature-of-Suit string."""
    if not isinstance(nos, str): return "other"
    s = nos.lower()
    for k, v in [("civil right","civil rights"),("contract","contract"),("tort","tort"),
                 ("employ","employment"),("labor","employment"),("administrativ","administrative"),
                 ("family","family")]:
        if k in s: return v
    return "other"

In [2]:
D = "../data/coded/"
full = pd.read_csv(D+"controls_all_fulltext.csv")   # case_name, year, nature_of_suit, text_for_coding, ...
llm  = pd.read_csv(D+"controls_llm_coded.csv")       # case_name, llm_severity, llm_eligible, ...

# --- de-duplicate, with before/after counts ---
n_full = len(full)
full = full.drop_duplicates(subset=["text_for_coding"]).copy()
print(f"controls: {n_full} rows -> {len(full)} after de-duplicating identical opinions")
n_llm = len(llm)
llm = llm.drop_duplicates("case_name", keep="first")
print(f"llm codes: {n_llm} rows -> {len(llm)} after de-duplicating case_name")

# --- MERGE: attach LLM codes to controls. print before/after + unmatched-key check ---
n_before = len(full)
ctrl = full.merge(llm[["case_name","llm_severity","llm_eligible"]], on="case_name", how="left")
print(f"\nmerge (left join on case_name): {n_before} rows before -> {len(ctrl)} after "
      f"(row count preserved: {len(ctrl) == n_before})")
matched = int(ctrl["llm_severity"].notna().sum())
print(f"  matched an LLM code: {matched} | unmatched (dropped): {len(ctrl) - matched}")

# --- keep eligible, coded controls ---
ctrl = ctrl[ctrl["llm_eligible"] == True].dropna(subset=["llm_severity"]).copy()
print(f"  after dropping ineligible + unmatched: {len(ctrl)} controls")

controls = pd.DataFrame({
    "case_name": ctrl["case_name"], "year": ctrl.get("year"), "ai": 0,
    "severity": ctrl["llm_severity"].astype(int),
    "pro_se": np.nan, "federal": np.nan,
    "field": ctrl["nature_of_suit"].map(field_from_nos)})

controls: 797 rows -> 663 after de-duplicating identical opinions
llm codes: 797 rows -> 738 after de-duplicating case_name

merge (left join on case_name): 663 rows before -> 663 after (row count preserved: True)
  matched an LLM code: 663 | unmatched (dropped): 0
  after dropping ineligible + unmatched: 321 controls


In [3]:
# --- AI arm ---
ai = pd.read_csv(D+"analysis_data_coded.csv")
ai_arm = pd.DataFrame({
    "case_name": ai.get("Case Name"), "year": ai.get("year"), "ai": 1,
    "severity": ai["severity"].astype(int), "pro_se": ai.get("pro_se"),
    "federal": ai.get("federal"), "field": ai.get("field")})

# --- CONCAT the two arms: print before/after with a sum check ---
n_ai, n_ctrl = len(ai_arm), len(controls)
pooled = pd.concat([ai_arm, controls], ignore_index=True)
print(f"concat: AI={n_ai} + control={n_ctrl} -> pooled={len(pooled)} "
      f"(sum check: {len(pooled) == n_ai + n_ctrl})")

pooled.to_csv(D+"pooled_coded.csv", index=False)
print(f"wrote {D}pooled_coded.csv")
print("\nseverity by arm:")
print(pooled.groupby("ai")["severity"].value_counts().unstack().fillna(0).astype(int).to_string())
print("\nmean severity by arm:", pooled.groupby("ai")["severity"].mean().round(2).to_dict())
print("\nNow run 04 (analyze) for the AI figures + pooled regression.")

concat: AI=1279 + control=321 -> pooled=1600 (sum check: True)
wrote ../data/coded/pooled_coded.csv

severity by arm:
severity    0    1    2    3    4
ai                               
0         203   15   20   28   55
1         256  498  217  135  173

mean severity by arm: {0: 1.12, 1: 1.59}

Now run 04 (analyze) for the AI figures + pooled regression.
